In [ ]:
from sklearn.calibration import calibration_curve


def plot_score_distribution(data, score_col, target_col, title):
    plt.figure(figsize=(10, 5))
    sns.histplot(
        data[data[target_col] == 0][score_col],
        bins=100,
        stat="density",
        label="target=0",
        alpha=0.5,
    )
    sns.histplot(
        data[data[target_col] == 1][score_col],
        bins=100,
        stat="density",
        label="target=1",
        alpha=0.5,
    )
    plt.title(title)
    plt.xlabel(score_col)
    plt.ylabel("Density")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def plot_target_rate_by_decile(decile_df, title):
    plot_df = decile_df.reset_index().sort_values("score_decile")

    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_df, x="score_decile", y="target_rate", color="steelblue")
    plt.title(title)
    plt.xlabel("Score decile")
    plt.ylabel("Target rate")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_lift_by_decile(decile_df, base_rate, title):
    plot_df = decile_df.reset_index().copy()
    plot_df["lift"] = plot_df["target_rate"] / base_rate
    plot_df = plot_df.sort_values("score_decile")

    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_df, x="score_decile", y="lift", color="darkorange")
    plt.axhline(1, color="black", linestyle="--")
    plt.title(title)
    plt.xlabel("Score decile")
    plt.ylabel("Lift vs average target rate")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_calibration(data, score_col, target_col, title):
    prob_true, prob_pred = calibration_curve(
        data[target_col],
        data[score_col],
        n_bins=10,
        strategy="quantile",
    )

    plt.figure(figsize=(6, 6))
    plt.plot(prob_pred, prob_true, marker="o", label="model")
    plt.plot([0, 1], [0, 1], linestyle="--", color="black", label="perfect")
    plt.title(title)
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed target rate")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def plot_segment_target_rate(segment_df, title):
    plot_df = segment_df.sort_values("target_rate", ascending=False)

    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_df, x="segment", y="target_rate", color="steelblue")
    plt.title(title)
    plt.xlabel("Segment")
    plt.ylabel("Target rate")
    plt.xticks(rotation=30)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_segment_metrics(segment_df, title):
    plot_df = segment_df.melt(
        id_vars=["segment"],
        value_vars=["pr_auc", "roc_auc"],
        var_name="metric",
        value_name="value",
    )

    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_df, x="segment", y="value", hue="metric")
    plt.title(title)
    plt.xlabel("Segment")
    plt.ylabel("Metric value")
    plt.xticks(rotation=30)
    plt.grid(axis="y", alpha=0.3)
    plt.show()

In [ ]:
print("OOF plots")

plot_score_distribution(
    df,
    score,
    target,
    "OOF score distribution by target",
)

plot_target_rate_by_decile(
    deciles,
    "OOF target rate by score decile",
)

plot_lift_by_decile(
    deciles,
    base_rate=df[target].mean(),
    title="OOF lift by score decile",
)

plot_calibration(
    df,
    score,
    target,
    "OOF calibration curve",
)

plot_segment_target_rate(
    segment_metrics,
    "OOF target rate by DAC segment",
)

plot_segment_metrics(
    segment_metrics,
    "OOF model quality by DAC segment",
)

In [ ]:
if "oot_df" in globals() and "oot_score_col" in globals() and "oot_deciles" in globals():
    print("OOT plots")

    plot_score_distribution(
        oot_df,
        oot_score_col,
        target,
        "OOT score distribution by target",
    )

    plot_target_rate_by_decile(
        oot_deciles,
        "OOT target rate by score decile",
    )

    plot_lift_by_decile(
        oot_deciles,
        base_rate=oot_df[target].mean(),
        title="OOT lift by score decile",
    )

    plot_calibration(
        oot_df,
        oot_score_col,
        target,
        "OOT calibration curve",
    )

    if "oot_segment_metrics" in globals() and len(oot_segment_metrics) > 0:
        plot_segment_target_rate(
            oot_segment_metrics,
            "OOT target rate by DAC segment",
        )

        plot_segment_metrics(
            oot_segment_metrics,
            "OOT model quality by DAC segment",
        )
else:
    print("OOT was not calculated, plots skipped.")

In [ ]:
comparison_rows = []

comparison_rows.append({
    "dataset": "OOF",
    "target_rate": df[target].mean(),
    "pr_auc": sk_average_precision_score(df[target], df[score]),
    "roc_auc": sk_roc_auc_score(df[target], df[score]),
    "ap_gain": sk_average_precision_score(df[target], df[score]) - df[target].mean(),
    "top_decile_target_rate": deciles.loc[10, "target_rate"] if 10 in deciles.index else np.nan,
    "top_decile_lift": (
        deciles.loc[10, "target_rate"] / df[target].mean()
        if 10 in deciles.index else np.nan
    ),
    "brier": sk_brier_score_loss(df[target], df[score]),
})

if "oot_df" in globals() and "oot_score_col" in globals() and "oot_deciles" in globals():
    comparison_rows.append({
        "dataset": "OOT",
        "target_rate": oot_df[target].mean(),
        "pr_auc": sk_average_precision_score(oot_df[target], oot_df[oot_score_col]),
        "roc_auc": sk_roc_auc_score(oot_df[target], oot_df[oot_score_col]),
        "ap_gain": sk_average_precision_score(oot_df[target], oot_df[oot_score_col]) - oot_df[target].mean(),
        "top_decile_target_rate": oot_deciles.loc[10, "target_rate"] if 10 in oot_deciles.index else np.nan,
        "top_decile_lift": (
            oot_deciles.loc[10, "target_rate"] / oot_df[target].mean()
            if 10 in oot_deciles.index else np.nan
        ),
        "brier": sk_brier_score_loss(oot_df[target], oot_df[oot_score_col]),
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)